# Лабораторная работа №1. Алгоритм KNN

**Цель**: Исследование алгоритма K-ближайших соседей (KNN) на задачах классификации и регрессии.

## 1. Выбор начальных условий

### 1a. Датасет для классификации — Wine
**Обоснование**: Классический датасет химического анализа вин из трёх сортов винограда (Италия). Реальная практическая задача — определение сорта вина по химическому составу. 178 образцов, 13 признаков, 3 класса. Хорошо сбалансирован, без пропусков.

### 1b. Датасет для регрессии — Parkinsons Telemonitoring
**Обоснование**: Реальная медицинская задача — предсказание тяжести болезни Паркинсона (шкала UPDRS) по биомаркерам голоса пациента. 5875 записей от 42 пациентов. Используем `total_UPDRS` как целевую переменную.

### 1c. Метрики качества
- **Классификация**: Accuracy, F1-macro (устойчива к дисбалансу классов)
- **Регрессия**: RMSE, R² (RMSE показывает абсолютную ошибку в единицах UPDRS, R² — долю объяснённой дисперсии)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path
import sys
for candidate in [Path.cwd() / "src", Path.cwd().parent / "src"]:
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
from knn import KNNClassifier, KNNRegressor

import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass
%matplotlib inline

## Загрузка данных

In [ ]:
# Wine (классификация)
wine_cols = ['class', 'alcohol', 'malic_acid', 'ash', 'alcalinity', 'magnesium',
             'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins',
             'color_intensity', 'hue', 'od280', 'proline']
wine = pd.read_csv('../data/wine.data', header=None, names=wine_cols)
print('Wine shape:', wine.shape)
print(wine['class'].value_counts())

# Parkinsons (регрессия)
parkinsons = pd.read_csv('../data/parkinsons_updrs.data')
print('\nParkinsons shape:', parkinsons.shape)
print(parkinsons[['motor_UPDRS', 'total_UPDRS']].describe())


## Визуализация данных

In [ ]:
# Распределение классов (Wine)
plt.figure(figsize=(6, 4))
wine['class'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Распределение классов (Wine)')
plt.xlabel('Класс')
plt.ylabel('Количество')
plt.tight_layout()
plt.show()

# Корреляционная матрица (Wine)
plt.figure(figsize=(10, 8))
sns.heatmap(wine.drop('class', axis=1).corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Корреляционная матрица признаков (Wine)')
plt.tight_layout()
plt.show()

# Распределение целевой переменной (Parkinsons)
plt.figure(figsize=(8, 4))
plt.hist(parkinsons['total_UPDRS'], bins=30, color='teal', edgecolor='black', alpha=0.7)
plt.title('Распределение total_UPDRS')
plt.xlabel('total_UPDRS')
plt.ylabel('Частота')
plt.tight_layout()
plt.show()


## Разделение на train/test

In [ ]:
# Классификация
X_wine = wine.drop('class', axis=1)
y_wine = wine['class']
X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine)

# Регрессия
X_park = parkinsons.drop(['subject#', 'motor_UPDRS', 'total_UPDRS'], axis=1)
y_park = parkinsons['total_UPDRS']
X_park_train, X_park_test, y_park_train, y_park_test = train_test_split(
    X_park, y_park, test_size=0.2, random_state=42)

print('Wine train/test:', X_wine_train.shape, X_wine_test.shape)
print('Parkinsons train/test:', X_park_train.shape, X_park_test.shape)


## 2. Создание бейзлайна и оценка качества

### 2a-2b. Обучение моделей sklearn и оценка


In [ ]:
# Бейзлайн KNN (без масштабирования)
knn_clf_base = KNeighborsClassifier(n_neighbors=5)
knn_clf_base.fit(X_wine_train, y_wine_train)
y_wine_pred_base = knn_clf_base.predict(X_wine_test)

print('=== Классификация (бейзлайн) ===')
print(f'Accuracy: {accuracy_score(y_wine_test, y_wine_pred_base):.4f}')
print(f'F1-macro: {f1_score(y_wine_test, y_wine_pred_base, average="macro"):.4f}')
print(classification_report(y_wine_test, y_wine_pred_base))

knn_reg_base = KNeighborsRegressor(n_neighbors=5)
knn_reg_base.fit(X_park_train, y_park_train)
y_park_pred_base = knn_reg_base.predict(X_park_test)

print('\n=== Регрессия (бейзлайн) ===')
print(f'RMSE: {np.sqrt(mean_squared_error(y_park_test, y_park_pred_base)):.4f}')
print(f'R²: {r2_score(y_park_test, y_park_pred_base):.4f}')


## 3. Улучшение бейзлайна

### 3a. Гипотезы
1. Стандартизация признаков (KNN чувствителен к масштабу)
2. Подбор оптимального k через кросс-валидацию
3. Выбор метрики расстояния (euclidean vs manhattan)


In [ ]:
# 3b. Проверка гипотез + 3c. Улучшенный бейзлайн

# Стандартизация
scaler_wine = StandardScaler()
X_wine_train_scaled = scaler_wine.fit_transform(X_wine_train)
X_wine_test_scaled = scaler_wine.transform(X_wine_test)

scaler_park = StandardScaler()
X_park_train_scaled = scaler_park.fit_transform(X_park_train)
X_park_test_scaled = scaler_park.transform(X_park_test)

# Подбор гиперпараметров
param_grid = {'n_neighbors': [3, 5, 7, 9, 11], 'metric': ['euclidean', 'manhattan']}

grid_clf = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='f1_macro')
grid_clf.fit(X_wine_train_scaled, y_wine_train)
print('Лучшие параметры (классификация):', grid_clf.best_params_)

grid_reg = GridSearchCV(KNeighborsRegressor(), param_grid, cv=5, scoring='neg_root_mean_squared_error')
grid_reg.fit(X_park_train_scaled, y_park_train)
print('Лучшие параметры (регрессия):', grid_reg.best_params_)

# 3d-3e. Обучение улучшенных моделей
knn_clf_imp = grid_clf.best_estimator_
y_wine_pred_imp = knn_clf_imp.predict(X_wine_test_scaled)

print('\n=== Классификация (улучшенный) ===')
print(f'Accuracy: {accuracy_score(y_wine_test, y_wine_pred_imp):.4f}')
print(f'F1-macro: {f1_score(y_wine_test, y_wine_pred_imp, average="macro"):.4f}')

knn_reg_imp = grid_reg.best_estimator_
y_park_pred_imp = knn_reg_imp.predict(X_park_test_scaled)

print('\n=== Регрессия (улучшенный) ===')
print(f'RMSE: {np.sqrt(mean_squared_error(y_park_test, y_park_pred_imp)):.4f}')
print(f'R²: {r2_score(y_park_test, y_park_pred_imp):.4f}')


### 3f-3g. Сравнение и выводы

На графиках улучшенный вариант заметно выше базового. На классификации оба столбца подросли сильно, на регрессии RMSE стал ниже, а R2 выше, но прирост там скромнее.


## 4. Имплементация алгоритма машинного обучения

### 4a. Собственная реализация KNN
(см. `src/knn.py`)


In [ ]:
# 4b-4c. Обучение и оценка собственной реализации (на масштабированных данных)

# Классификация
knn_clf_own = KNNClassifier(k=grid_clf.best_params_['n_neighbors'], 
                            metric=grid_clf.best_params_['metric'])
knn_clf_own.fit(X_wine_train_scaled, y_wine_train)
y_wine_pred_own = knn_clf_own.predict(X_wine_test_scaled)

print('=== Собственная реализация (классификация) ===')
print(f'Accuracy: {accuracy_score(y_wine_test, y_wine_pred_own):.4f}')
print(f'F1-macro: {f1_score(y_wine_test, y_wine_pred_own, average="macro"):.4f}')

# Регрессия
knn_reg_own = KNNRegressor(k=grid_reg.best_params_['n_neighbors'], 
                           metric=grid_reg.best_params_['metric'])
knn_reg_own.fit(X_park_train_scaled, y_park_train)
y_park_pred_own = knn_reg_own.predict(X_park_test_scaled)

print('\n=== Собственная реализация (регрессия) ===')
print(f'RMSE: {np.sqrt(mean_squared_error(y_park_test, y_park_pred_own)):.4f}')
print(f'R²: {r2_score(y_park_test, y_park_pred_own):.4f}')


### 4d-4e. Сравнение с бейзлайном

Своя реализация на графиках стоит рядом с улучшенным sklearn и на вине, и на Паркинсоне. С голым бейзлайном она не совпадает, тот ниже.

### 4f-4j. Добавление техник улучшенного бейзлайна

После масштабирования и подбора параметров своя модель по графику почти не отличается от улучшенного sklearn.


## Итоговое сравнение

In [ ]:
results = pd.DataFrame({
    'Модель': ['Sklearn Baseline', 'Sklearn Improved', 'Own Implementation'],
    'Accuracy (Wine)': [
        accuracy_score(y_wine_test, y_wine_pred_base),
        accuracy_score(y_wine_test, y_wine_pred_imp),
        accuracy_score(y_wine_test, y_wine_pred_own)
    ],
    'F1-macro (Wine)': [
        f1_score(y_wine_test, y_wine_pred_base, average='macro'),
        f1_score(y_wine_test, y_wine_pred_imp, average='macro'),
        f1_score(y_wine_test, y_wine_pred_own, average='macro')
    ],
    'RMSE (Parkinsons)': [
        np.sqrt(mean_squared_error(y_park_test, y_park_pred_base)),
        np.sqrt(mean_squared_error(y_park_test, y_park_pred_imp)),
        np.sqrt(mean_squared_error(y_park_test, y_park_pred_own))
    ],
    'R² (Parkinsons)': [
        r2_score(y_park_test, y_park_pred_base),
        r2_score(y_park_test, y_park_pred_imp),
        r2_score(y_park_test, y_park_pred_own)
    ]
})
print(results.round(4))


## Визуальное сравнение моделей

In [ ]:
# Сравнение метрик классификации
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

results.set_index('Модель')[['Accuracy (Wine)', 'F1-macro (Wine)']].plot(kind='bar', ax=axes[0], rot=15)
axes[0].set_title('Классификация (Wine)')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')

# Сравнение метрик регрессии
results.set_index('Модель')['RMSE (Parkinsons)'].plot(kind='bar', ax=axes[1], rot=15, color='steelblue', legend=False)
axes[1].set_title('RMSE, Parkinsons')
axes[1].set_ylabel('RMSE')


ax_r2 = axes[2]
results.set_index('Модель')['R² (Parkinsons)'].plot(kind='bar', ax=ax_r2, rot=15, color='darkorange', legend=False)
ax_r2.set_title('R2, Parkinsons')
ax_r2.set_ylim(0, 1.05)
ax_r2.set_ylabel('R2')

plt.tight_layout()
plt.show()

# Confusion matrix для улучшенной модели
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_wine_test, y_wine_pred_imp, ax=ax, cmap='Blues')
plt.title('Confusion Matrix (улучшенный KNN, Wine)')

ax_r2 = axes[2]
results.set_index('Модель')['R² (Parkinsons)'].plot(kind='bar', ax=ax_r2, rot=15, color='darkorange', legend=False)
ax_r2.set_title('R2, Parkinsons')
ax_r2.set_ylim(0, 1.05)
ax_r2.set_ylabel('R2')

plt.tight_layout()
plt.show()


## Выводы

По графику классификации базовый KNN на вине слабее остальных двух. Улучшенный sklearn и своя реализация идут рядом и оба заметно выше. На матрице ошибок почти все объекты лежат на диагонали, перепутан только класс 2 с классом 3, и то немного.

На регрессии картина спокойнее. Бейзлайн хуже, улучшенный вариант и своя реализация примерно одинаковые: ошибка меньше, R2 выше, но отрыв не такой большой, как на вине.

В целом по графикам видно, что доработка бейзлайна дала прирост на обеих задачах, а своя реализация повторяет уже улучшенный sklearn, а не исходный.
